In [1]:
from pathlib import Path
from typing import Iterable
import sys
import os
import pandas as pd
from shared import normalize_column_names, project_path

# Load BLS data

This notebook takes the BLS OEWS bulk time-series files [that you have downloaded from here](https://download.bls.gov/pub/time.series/oe), filters them to current national/state/metro occupation employment estimates, and writes a stable processed CSV for later stages.

Manually save the following OEWS files to the `data_dir` directory:
- oe.series
- oe.data.0.Current
- oe.area
- oe.areatype
- oe.datatype
- oe.occupation
- oe.footnote
- oe.txt

I know, I know... wtf is that `data_dir` path? It's so stupid because the stinky AI version of the code did this part way better than I did. Yes I have some shame but I'm happy to learn from the robots. But the data are HUGE so I'm not f-ing downloading them again damnit.

## Establish settings

In [2]:
#RAW_DIR = project_path('data', 'raw', 'bls_oews')
OUTPUT_PATH = project_path('data', 'processed', 'bls_oews_current_employment.csv')
data_dir = Path('../../../../../_AI-TESTING/ai-job-exposure/data/raw/bls_oews/')

## Setup helper functions

These helpers are only used in this ETL notebook, so they live here rather than in `shared.py`.

In [3]:
# BLS_BASE_URL = 'https://download.bls.gov/pub/time.series/oe'
# BLS_REQUEST_HEADERS = {
#     'User-Agent': 'ai-job-exposure reporting project (contact: local newsroom data analysis)',
# }
BLS_FILES = [
    'oe.series',
    'oe.data.0.Current',
    'oe.area',
    'oe.areatype',
    'oe.datatype',
    'oe.occupation',
    'oe.footnote',
    'oe.txt',
]

CROSS_INDUSTRY_CODE = '000000'
ALL_OCCUPATIONS_CODE = '000000'
EMPLOYMENT_DATATYPE_CODE = '01'
ANNUAL_PERIOD = 'A01'
REPORTING_AREATYPES = ['N', 'S', 'M']


def read_bls_table(data_dir: Path, filename: str) -> pd.DataFrame:
    path = data_dir / f'{filename}.txt'
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run this notebook download step first.')
    return pd.read_csv(path, sep='	', dtype=str)


def read_filtered_series(data_dir: Path, chunksize: int = 250_000) -> pd.DataFrame:
    path = data_dir / 'oe.series.txt'
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run this notebook download step first.')

    chunks = []
    for chunk in pd.read_csv(path, sep='	', dtype=str, chunksize=chunksize):
        chunk = normalize_column_names(chunk)
        filtered = chunk[
            chunk['areatype_code'].isin(REPORTING_AREATYPES)
            & chunk['industry_code'].eq(CROSS_INDUSTRY_CODE)
            & chunk['datatype_code'].eq(EMPLOYMENT_DATATYPE_CODE)
        ].copy()
        if not filtered.empty:
            chunks.append(filtered)

    if not chunks:
        raise ValueError('No matching cross-industry employment series were found in oe.series.')
    return pd.concat(chunks, ignore_index=True)


def read_filtered_current_data(data_dir: Path, series_ids: set[str], chunksize: int = 500_000) -> pd.DataFrame:
    path = data_dir / 'oe.data.0.Current.txt'
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run this notebook download step first.')

    chunks = []
    for chunk in pd.read_csv(path, sep='	', dtype=str, chunksize=chunksize):
        chunk = normalize_column_names(chunk)
        filtered = chunk[chunk['series_id'].isin(series_ids) & chunk['period'].eq(ANNUAL_PERIOD)].copy()
        if not filtered.empty:
            chunks.append(filtered)

    if not chunks:
        raise ValueError('No current annual observations matched the filtered OEWS series.')
    return pd.concat(chunks, ignore_index=True)


def format_soc_code(occupation_code: str | float | None) -> str | None:
    if occupation_code is None or pd.isna(occupation_code):
        return None
    raw = str(occupation_code).strip().replace('-', '')
    if not raw or raw == ALL_OCCUPATIONS_CODE or len(raw) != 6 or not raw.isdigit():
        return None
    return f'{raw[:2]}-{raw[2:]}'


def read_current_employment(data_dir: Path) -> pd.DataFrame:
    series = read_filtered_series(data_dir)
    data = read_filtered_current_data(data_dir, set(series['series_id']))
    area = normalize_column_names(read_bls_table(data_dir, 'oe.area'))
    occupation = normalize_column_names(read_bls_table(data_dir, 'oe.occupation'))
    datatype = normalize_column_names(read_bls_table(data_dir, 'oe.datatype'))
    areatype = normalize_column_names(read_bls_table(data_dir, 'oe.areatype'))

    data['employment'] = pd.to_numeric(data['value'], errors='coerce')
    merged = data.merge(
        series[[
            'series_id', 'seasonal', 'areatype_code', 'industry_code', 'occupation_code',
            'datatype_code', 'state_code', 'area_code', 'series_title',
        ]],
        on='series_id',
        how='inner',
        validate='many_to_one',
    )
    merged = merged.merge(area[['area_code', 'area_name']], on='area_code', how='left', validate='many_to_one')
    merged = merged.merge(areatype[['areatype_code', 'areatype_name']], on='areatype_code', how='left', validate='many_to_one')
    merged = merged.merge(occupation[['occupation_code', 'occupation_name']], on='occupation_code', how='left', validate='many_to_one')
    merged = merged.merge(datatype[['datatype_code', 'datatype_name']], on='datatype_code', how='left', validate='many_to_one')

    merged['year'] = pd.to_numeric(merged['year'], errors='coerce').astype('Int64')
    merged['soc_code'] = merged['occupation_code'].map(format_soc_code)
    merged['is_all_occupations'] = merged['occupation_code'].eq(ALL_OCCUPATIONS_CODE)
    merged['has_released_employment'] = merged['employment'].notna()

    columns = [
        'series_id', 'year', 'period', 'areatype_code', 'areatype_name', 'state_code',
        'area_code', 'area_name', 'occupation_code', 'soc_code', 'occupation_name',
        'employment', 'footnote_codes', 'series_title', 'is_all_occupations',
        'has_released_employment',
    ]
    return merged[columns].sort_values(['areatype_code', 'area_name', 'occupation_code']).reset_index(drop=True)

## Parse and export

The parser streams the two largest files and keeps only cross-industry employment rows for national, state and metro geographies.

In [4]:
employment = read_current_employment(data_dir)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
employment.to_csv(OUTPUT_PATH, index=False)

print(f'Wrote {len(employment):,} rows to {OUTPUT_PATH}')
employment.head()

Wrote 236,799 rows to /Users/alexandra.kanik/DEVHUB/projects/_ACTIVE/ai-job-adaptation/AI-job-adaptation/analysis/data/processed/bls_oews_current_employment.csv


,series_id,year,period,areatype_code,areatype_name,state_code,area_code,area_name,occupation_code,soc_code,occupation_name,employment,footnote_codes,series_title,is_all_occupations,has_released_employment
0,OEUM001018000000000000001,2025,A01,M,Metropolitan or nonmetropolitan area,48,0010180,"Abilene, TX",000000,NaN,All Occupations,75070.0,NaN,Employment for All Occupations in All Industri...,True,True
1,OEUM001018000000011000001,2025,A01,M,Metropolitan or nonmetropolitan area,48,0010180,"Abilene, TX",110000,11-0000,Management Occupations,5470.0,NaN,Employment for Management Occupations in All I...,False,True
2,OEUM001018000000011101101,2025,A01,M,Metropolitan or nonmetropolitan area,48,0010180,"Abilene, TX",111011,11-1011,Chief Executives,40.0,NaN,Employment for Chief Executives in All Industr...,False,True
3,OEUM001018000000011102101,2025,A01,M,Metropolitan or nonmetropolitan area,48,0010180,"Abilene, TX",111021,11-1021,General and Operations Managers,2120.0,NaN,Employment for General and Operations Managers...,False,True
4,OEUM001018000000011202101,2025,A01,M,Metropolitan or nonmetropolitan area,48,0010180,"Abilene, TX",112021,11-2021,Marketing Managers,140.0,NaN,Employment for Marketing Managers in All Indus...,False,True


## Checks

These checks give an editor a quick read on geography coverage, year coverage and missing employment values.

In [5]:
print('Rows by geography type:')
print(employment.groupby(['areatype_code', 'areatype_name']).size().to_string())

print('\nYears in file:')
print(employment['year'].value_counts(dropna=False).sort_index().to_string())

print('\nReleased employment share:')
print(employment['has_released_employment'].mean())

Rows by geography type:
areatype_code  areatype_name                       
M              Metropolitan or nonmetropolitan area    198287
N              National                                  1104
S              State                                    37408

Years in file:
year
2025    236799

Released employment share:
0.9788428160591894
